#**Treinar o Classificador Supervisionado e Definir o Limiar (Threshold)**


 # Entendendo o Conceito

## O que é o Classificador?

É um modelo de Inteligência Artificial que "lê" o resumo higienizado de um artigo (`resumo_limpo`) e aprende a adivinhar a qual `categoria_projeto` ele pertence (ex: Inteligência Artificial, DevOps, Mobile, etc.).

## Como ele aprende?

Ele usa os **5.963 artigos** tratados na Semana 1. Ele analisa quais palavras aparecem com mais frequência em cada categoria.

## O que é o Threshold (Limiar de Confiança)?

Quando a IA analisa um texto novo recebido pela API do Backend, ela dá uma nota de certeza de **0% a 100%** (ex: "Tenho 85% de certeza que isso é DevOps").

- Se a certeza for $\ge 70\%$ (0.70): O artigo é aprovado automaticamente.
- Se a certeza for $< 70\%$: O artigo é enviado para a Fila de Moderação para um humano revisar (conceito de Human-in-the-Loop).

In [3]:
# ==============================================================================
# PASSO 1: TREINAMENTO DO CLASSIFICADOR E GERAÇÃO DOS ARTEFATOS PARA O BACKEND
# ==============================================================================

import json
import pandas as pd
import joblib

# Bibliotecas de Aprendizado de Máquina (scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

print("1. Carregando a base de dados tratada...")
# Carrega o JSON da Semana 1 que você gerou
df = pd.read_json("dataset_techmind_evolution.json")

# Define as variáveis de Entrada (X) e Saída/Rótulo (y)
X = df["resumo_limpo"]
y = df["categoria_projeto"]

# ------------------------------------------------------------------------------
# 2. Divisão de Treino e Teste (80% para treinar a IA, 20% para testar a acurácia)
# ------------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Dados de Treino: {len(X_train)} artigos | Dados de Teste: {len(X_test)} artigos")

# ------------------------------------------------------------------------------
# 3. Construção do Pipeline do Modelo
# - TF-IDF: Converte texto em números atribuindo peso às palavras raras/importantes.
# - SGDClassifier (com log_loss): Modelo rápido e eficiente que gera probabilidades (predict_proba).
# ------------------------------------------------------------------------------
print("2. Treinando o modelo de Inteligência Artificial...")

model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', SGDClassifier(loss='log_loss', max_iter=1000, random_state=42))
])

# Treina o modelo com os dados de treino
model_pipeline.fit(X_train, y_train)

# ------------------------------------------------------------------------------
# 4. Avaliação do Desempenho
# ------------------------------------------------------------------------------
print("\n=== AVALIAÇÃO DO MODELO ===")
y_pred = model_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

# ------------------------------------------------------------------------------
# 5. Testando a Lógica do Limiar (Threshold = 0.70)
# ------------------------------------------------------------------------------
def inferir_com_threshold(texto_limpo, pipeline_modelo, limiar=0.70):
    """
    Função que simula o comportamento que o Backend executará na API.
    """
    # Obtém as probabilidades calculadas pelo modelo para cada categoria
    probabilidades = pipeline_modelo.predict_proba([texto_limpo])[0]

    # Pega a maior probabilidade e a categoria correspondente
    maior_probabilidade = probabilidades.max()
    indice_categoria = probabilidades.argmax()
    categoria_sugerida = pipeline_modelo.classes_[indice_categoria]

    if maior_probabilidade >= limiar:
        status = "Aprovado"
    else:
        status = "Pendente" # Vai para a Fila de Moderação Humana

    return {
        "categoria_sugerida": categoria_sugerida,
        "confianca": round(float(maior_probabilidade), 4),
        "status": status
    }

# Teste rápido com um resumo de teste
exemplo_teste = X_test.iloc[0]
resultado_teste = inferir_com_threshold(exemplo_teste, model_pipeline)
print("Exemplo de Saída da Inferência:")
print(resultado_teste)

# ------------------------------------------------------------------------------
# 6. EXPORTAÇÃO DOS ENTREGÁVEIS PARA O BACKEND
# ------------------------------------------------------------------------------
print("\n3. Exportando artefatos para a Squad de Backend...")

# Salva o arquivo binário do modelo treinado
joblib.dump(model_pipeline, 'classificador_techmind.pkl')

# Salva as configurações de regra de negócio em um arquivo JSON
config_backend = {
    "limiar_confianca": 0.70,
    "categorias_suportadas": list(model_pipeline.classes_),
    "algoritmo": "TF-IDF + SGDClassifier (log_loss)",
    "data_treinamento": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open('config_classificador.json', 'w', encoding='utf-8') as f:
    json.dump(config_backend, f, ensure_ascii=False, indent=4)

print("✅ SUCESSO! Arquivos 'classificador_techmind.pkl' e 'config_classificador.json' gerados com sucesso!")

1. Carregando a base de dados tratada...
Dados de Treino: 4770 artigos | Dados de Teste: 1193 artigos
2. Treinando o modelo de Inteligência Artificial...

=== AVALIAÇÃO DO MODELO ===
                         precision    recall  f1-score   support

                Backend       0.78      0.58      0.67       120
         Banco de Dados       0.79      0.83      0.81       120
         Cibersegurança       0.56      0.72      0.64       120
        Cloud Computing       0.79      0.74      0.77       120
           Data Science       0.44      0.50      0.47       120
                 Devops       0.86      0.81      0.83       113
               Frontend       0.75      0.73      0.74       120
Inteligencia artificial       0.47      0.47      0.47       120
                 Mobile       0.79      0.64      0.71       120
                  Redes       0.72      0.79      0.75       120

               accuracy                           0.68      1193
              macro avg       0.70 

#**Versão Otimizada**

# Alterações feitas

## 1. Juntar Título + Resumo (`titulo + resumo_limpo`)

**Por quê?**

Os títulos dos artigos contêm palavras de altíssimo valor conceitual (ex: "Kubernetes", "YOLO", "PostgreSQL"). Ao alimentar o modelo com o título e o resumo juntos, damos muito mais sinal para a IA decidir a categoria.

---

## 2. Melhorar a Vetorização (`TF-IDF` com `sublinear_tf=True` e `max_features=10000`)

### `max_features=10000`

Aumentamos o vocabulário das **5.000** para as **10.000** palavras/expressões mais importantes.

### `sublinear_tf=True`

Aplica uma escala logarítmica na contagem das palavras. Isso impede que uma palavra que se repete 20 vezes num mesmo texto afogue o peso das outras palavras importantes.

---

## 3. Substituir o Classificador por Regressão Logística Otimizada (`LogisticRegression`)

**Por quê?**

O `SGDClassifier` é ótimo para bases de dados gigantescas de milhões de linhas, mas para a nossa base de aproximadamente **6.000 artigos**, a `LogisticRegression(C=2.0)` consegue calcular as fronteiras de decisão entre as 10 categorias com muito mais precisão e estabilidade.

# 📊 1. Comparativo Geral: Salto na Acurácia Global

- **Modelo Antigo** (`SGDClassifier` + apenas Resumo + 5.000 termos): **68.00% de acurácia**.

- **Modelo Novo** (`LogisticRegression` + Título + Resumo + 10.000 termos): **70.91% de acurácia**.

- **Ganho Real:** **+2.91%** de ganho direto em acerto global.

In [5]:
# ==============================================================================
# PASSO 1 (VERSÃO OTIMIZADA): TREINAMENTO DO CLASSIFICADOR COM ALTA ACURÁCIA
# ==============================================================================

import json
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# 1. Carregando os dados
print("1. Carregando e enriquecendo a base de dados...")
df = pd.read_json("dataset_techmind_evolution.json")

# OTIMIZAÇÃO 1: Unir Título e Resumo para dar mais contexto à IA
df["texto_completo"] = df["titulo"].fillna("") + " " + df["resumo_limpo"].fillna("")

X = df["texto_completo"]
y = df["categoria_projeto"]

# Separação Treino/Teste (80/20) mantendo a reprodutibilidade
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Dados de Treino: {len(X_train)} artigos | Dados de Teste: {len(X_test)} artigos")

# 2. Pipeline Otimizado de IA
print("2. Treinando o modelo Otimizado de Inteligência Artificial...")

model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,      # Aumentado para 10.000 termos
        ngram_range=(1, 2),      # Analisa palavras soltas e pares
        sublinear_tf=True,       # Suavização logarítmica da frequência
        min_df=2                 # Descarta erros raríssimos
    )),
    ('clf', LogisticRegression(
        C=2.0,                   # Controle de regularização otimizado
        max_iter=1000,
        random_state=42
    ))
])

# Treinamento do modelo
model_pipeline.fit(X_train, y_train)

# 3. Avaliação no conjunto de teste
y_pred = model_pipeline.predict(X_test)
acuracia = accuracy_score(y_test, y_pred)

print("\n=== AVALIAÇÃO DO MODELO OTIMIZADO ===")
print(f"🎯 NOVA ACURÁCIA GLOBAL: {acuracia:.2%}\n")
print(classification_report(y_test, y_pred))

# ------------------------------------------------------------------------------
# 4. TESTE DE INFERÊNCIA COM LIMIAR DE CONFIANÇA (THRESHOLD = 0.70)
# ------------------------------------------------------------------------------
LIMIAR_CONFIANÇA = 0.70

def classificar_artigo(texto_artigo, pipeline_modelo):
    probabilidades = pipeline_modelo.predict_proba([texto_artigo])[0]
    classes = pipeline_modelo.classes_

    idx_max = np.argmax(probabilidades)
    categoria_sugerida = classes[idx_max]
    confianca = probabilidades[idx_max]

    status = "Aprovado" if confianca >= LIMIAR_CONFIANÇA else "Pendente"

    return {
        "categoria_sugerida": categoria_sugerida,
        "confianca": round(float(confianca), 4),
        "status": status
    }

# Teste com um exemplo
exemplo_teste = X_test.iloc[0]
resultado_teste = classificar_artigo(exemplo_teste, model_pipeline)
print("Exemplo de Saída da Inferência:")
print(resultado_teste)

# ------------------------------------------------------------------------------
# 5. EXPORTAÇÃO DOS ARTEFATOS
# ------------------------------------------------------------------------------
print("\n3. Exportando artefatos otimizados...")
joblib.dump(model_pipeline, 'classificador_techmind.pkl')

config_classificador = {
    "limiar_confianca": LIMIAR_CONFIANÇA,
    "categorias_suportadas": list(model_pipeline.classes_),
    "acuracia_teste": round(float(acuracia), 4),
    "modelo": "TfidfVectorizer + LogisticRegression(C=2.0)"
}

with open('config_classificador.json', 'w', encoding='utf-8') as f:
    json.dump(config_classificador, f, ensure_ascii=False, indent=4)

print("✅ SUCESSO! Artefatos 'classificador_techmind.pkl' e 'config_classificador.json' atualizados!")

1. Carregando e enriquecendo a base de dados...
Dados de Treino: 4770 artigos | Dados de Teste: 1193 artigos
2. Treinando o modelo Otimizado de Inteligência Artificial...

=== AVALIAÇÃO DO MODELO OTIMIZADO ===
🎯 NOVA ACURÁCIA GLOBAL: 70.91%

                         precision    recall  f1-score   support

                Backend       0.81      0.60      0.69       120
         Banco de Dados       0.80      0.83      0.82       120
         Cibersegurança       0.58      0.77      0.66       120
        Cloud Computing       0.83      0.76      0.79       120
           Data Science       0.46      0.53      0.49       120
                 Devops       0.92      0.81      0.86       113
               Frontend       0.81      0.82      0.82       120
Inteligencia artificial       0.50      0.54      0.52       120
                 Mobile       0.84      0.66      0.74       120
                  Redes       0.73      0.77      0.75       120

               accuracy                  

**Download artefatos gerados anteriormente**

In [6]:
from google.colab import files

files.download('classificador_techmind.pkl')
files.download('config_classificador.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 🔄 3. O Plano de Evolução para o Projeto

Para a entrega atual e apresentação, a estratégia é a seguinte:

## Fase Atual (Entrega do MVP / Projeto Prático)

**Manter:** `LogisticRegression(C=2.0)`

**Justificativa no Relatório:**

> "Optou-se pela Regressão Logística na versão inicial devido ao ganho de acurácia (70,91%) e estabilidade na fronteira de decisão para a amostra atual de ~6.000 artigos."

---

## Fase de Produção / Escala Futura (Roadmap de Arquitetura)

**Migração:**

Quando a base ultrapassar **50.000 artigos** e a necessidade de re-treinamento contínuo em tempo real (**Active Learning**) for ativada, a equipe de Ciência de Dados fará a transição simples para o `SGDClassifier` usando `.partial_fit()`.